# 04 · Train shadow models

**Purpose.** Train shadow models with the identical recipe of each target: 16 per classical config (C2 at every checkpoint epoch), 4 per YOLO config (C6 at every listed checkpoint). Each shadow trains on its IN set and is evaluated on its OUT set; both come only from the shadow pool.

**Inputs.** Feature cache, split file, GPU for the YOLO shadows.

**Outputs.** `models` rows with role = shadow, `shadow_membership` rows (IN = 1, OUT = 0), probability caches.

**Syllabus topics.** Supervised classification; the shadow-model method of Shokri et al.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from mia.training import Workspace, run_jobs

ws = Workspace.open()
exp = ws.exp
jobs = []
for cid in ("C1", "C2", "C3", "C4", "C5", "C6", "C7"):
    mcfg = exp.configs[cid]
    jobs += [(cid, exp.seeds_for(mcfg)[0], k) for k in range(exp.n_shadows(mcfg))]
print(len(jobs), "shadow training runs")
summary = run_jobs(ws, jobs)

In [ ]:
from mia import db

counts = db.query_df(ws.conn, """SELECT config_id, epoch, COUNT(*) AS n_shadows, AVG(train_acc) AS train_acc,
                                       AVG(test_acc) AS test_acc FROM models WHERE role = 'shadow'
                                GROUP BY config_id, epoch ORDER BY config_id, epoch""")
counts.round(3)

In [ ]:
print(f"Shadow runs: trained {summary['trained']}, skipped {summary['skipped']}, failed {summary['failed']}")
print("shadow_membership rows:", ws.conn.execute("SELECT COUNT(*) FROM shadow_membership").fetchone()[0])

## Results
The table lists how many shadows exist per recipe and checkpoint and confirms that shadow accuracies mirror the targets', which is what lets the attack transfer.